In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from ast import parse
import re

# Define the regex pattern
answer_pattern = r'\b(Office|Garden|Kitchen|Bathroom|Hallway|Bedroom|\d)\b'

%config InlineBackend.figure_format='retina'

In [ ]:
%cd /workspace-SR004.nfs2/kurkin/long-vqa/

In [ ]:
questions = pd.read_csv("data/new_test_data.csv")

In [ ]:
print(questions.groupby(['Type'])['Answer'].value_counts().to_string())

In [ ]:
from typing import List, Dict, Tuple
from openai import AsyncOpenAI
from pydantic import BaseModel, NonNegativeInt, constr
from typing import Literal, Optional, Set
from tqdm.asyncio import tqdm_asyncio
import pandas as pd

prefix = ""

# Define the Pydantic models
room_names = ["Kitchen", "Bathroom", "Garden", "Office", "Bedroom", "Hallway"]
people_names = ["Nobody", "Daniel", "Mary", "Michael", "Sandra", "John"]

class RoomAnswer(BaseModel):
    reasoning: Optional[str]
    answer: Literal[*room_names]

class NumberAnswer(BaseModel):
    reasoning: Optional[str]
    answer: int

class PersonAnswer(BaseModel):
    reasoning: Optional[str]
    answer: Set[Literal[*people_names]]

In [ ]:
from ast import literal_eval

In [ ]:
from pydantic import BaseModel, ValidationError
from pandas import DataFrame
import pandas as pd
from pydantic import BaseModel, ValidationError
from typing import Optional, Set, Literal, Iterable
from pandas import DataFrame
import pandas as pd
import ast

# Function to safely evaluate a string into a Python object
def evaluate_string(s):
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return s  # Return the string as-is if evaluation fails

# Function to infer the correct Pydantic model based on the ground truth 'Answer'
def infer_model(answer):
    if isinstance(answer, int):
        return NumberAnswer
    elif isinstance(answer, str) and answer in room_names:
        return RoomAnswer
    elif isinstance(answer, (list, set)) and all(person in people_names for person in answer):
        return PersonAnswer
    elif isinstance(answer, str) and answer in people_names:
        return PersonAnswer
    else:
        raise ValueError(f"Unable to infer the correct model for the answer: {answer}")

# Function to normalize the answer (convert lists or single items to sets for PersonAnswer)
def normalize_answer(answer, model):
    if model == PersonAnswer:
        if isinstance(answer, (str, int)):
            return {answer}  # Convert single item to a set
        elif isinstance(answer, list):
            return set(answer)  # Convert list to a set
        elif isinstance(answer, set):
            return answer  # Already a set
    if model == NumberAnswer:
        if isinstance(answer, (str, Iterable)) and "Nobody" in answer:
            answer = 0
    return answer  # No normalization needed for other models

# Function to validate and compare answers
def validate_and_compare(row):
    try:
        # Evaluate the ground truth 'Answer' string into a Python object
        evaluated_answer = evaluate_string(row['Answer'])
        
        # Infer the model based on the evaluated ground truth 'Answer'
        model = infer_model(evaluated_answer)
        
        # Normalize the evaluated ground truth 'Answer'
        normalized_answer = normalize_answer(evaluated_answer, model)
        
        # Validate the ground truth 'Answer'
        validated_answer = model(reasoning=None, answer=normalized_answer)
        
        # Evaluate the 'Predicted_Answer' string into a Python object
        evaluated_prediction = evaluate_string(row['Final_Answer'])
        
        # Normalize the evaluated 'Predicted_Answer'
        normalized_prediction = normalize_answer(evaluated_prediction, model)
    
        # Validate the 'Predicted_Answer'
        validated_prediction = model(reasoning=None, answer=normalized_prediction)
        
        # Compare the validated answers
        return validated_answer.answer == validated_prediction.answer
    except ValidationError as e:
        print(f"Validation error: {e}")
        return False
    except ValueError as e:
        print(f"Value error: {e}")
        return False

# Example usage with a pandas DataFrame
data = {
    'Answer': ['"Kitchen"', '3', '{"Daniel", "Mary"}', '"Daniel"', '["Mary", "John"]'],
    'Final_Answer': ['"Kitchen"', '"4"', '{"Daniel", "John"}', '{"Nobody"}', '{"John", "Mary"}']
}

df = pd.DataFrame(data)

# Apply the validation function to each row
df['Validation_Result'] = df.apply(validate_and_compare, axis=1)

print(df)
print(df)

In [ ]:
schemas = {"room": RoomAnswer.model_json_schema(), "number": NumberAnswer.model_json_schema(), "person": PersonAnswer.model_json_schema()}

person_types = {"most_time_in_room", "compare_two_steps", "list_chars_in_room_at_step", "name_char_with_char_at_step"}


In [ ]:
from json_repair import repair_json

In [ ]:
import json_repair

In [ ]:
# Paths to answer files
answers = [
    "data/qa_pairs_answers_Qwen_Qwen2-VL-2B-Instruct.csv",
    "data/qa_pairs_answers_Qwen_Qwen2-VL-7B-Instruct.csv",
    "data/qa_pairs_answers_Qwen_Qwen2-VL-72B-Instruct.csv",
    "data/qa_pairs_answers_Qwen_Qwen2.5-VL-3B-Instruct.csv",
    "data/qa_pairs_answers_Qwen_Qwen2.5-VL-7B-Instruct.csv",
    "data/qa_pairs_answers_Qwen_Qwen2.5-VL-72B-Instruct.csv",
    # "data/qa_pairs_answers_Qwen_QVQ-72B-Preview.csv",
    # "data/qa_pairs_answers_OpenGVLab_InternVL2_5-1B.csv",
    # "data/qa_pairs_answers_OpenGVLab_InternVL2_5-1B-MPO.csv",
    # "data/qa_pairs_answers_OpenGVLab_InternVL2_5-2B.csv",
    # "data/qa_pairs_answers_OpenGVLab_InternVL2_5-2B-MPO.csv",
    # "data/qa_pairs_answers_OpenGVLab_InternVL2_5-4B.csv",
    # "data/qa_pairs_answers_OpenGVLab_InternVL2_5-4B-MPO.csv",
    # # "data/qa_pairs_answers_OpenGVLab_Mono-InternVL-2B.csv", # cross attention
    # "data/qa_pairs_answers_OpenGVLab_InternVL2_5-8B.csv",
    # "data/qa_pairs_answers_OpenGVLab_InternVL2_5-8B-MPO.csv",
    # "data/qa_pairs_answers_OpenGVLab_InternVL2_5-26B.csv", # очень низкие метрики
    # "data/qa_pairs_answers_OpenGVLab_InternVL2_5-26B-MPO.csv", # очень низкие метрики
    # "data/qa_pairs_answers_OpenGVLab_InternVL2_5-38B.csv",
    # "data/qa_pairs_answers_OpenGVLab_InternVL2_5-38B-MPO.csv",
    # "data/qa_pairs_answers_OpenGVLab_InternVL2_5-78B.csv",
    # "data/qa_pairs_answers_OpenGVLab_InternVL2_5-78B-MPO.csv",
    # "data/qa_pairs_answers_microsoft_Phi-3.5-vision-instruct.csv",
    # "data/qa_pairs_answers_openbmb_MiniCPM-V-2_6.csv",
    # # "data/qa_pairs_answers_HuggingFaceM4_Idefics3-8B-Llama3.csv", # низкие метрики
    # # "data/qa_pairs_answers_cambrian.csv", # пока непонятно в чём ключевое отличие
    # "data/qa_pairs_answers_unsloth_Llama-3.2-11B-Vision-Instruct.csv",
    # "data/qa_pairs_answers_llava-hf_llava-onevision-qwen2-7b-ov-hf.csv",
    # "data/qa_pairs_answers_llava-hf_llava-onevision-qwen2-7b-si-hf.csv",
]

In [ ]:
heatmap_data = []
all_answers = []

# Process each model's data
for path in answers:
    if not os.path.exists(path):
        continue
    # Read CSV
    df_answers = pd.read_csv(path)
    print(path, df_answers.shape)
    df_answers = df_answers[~df_answers['Predicted_Answer'].str.lower().str.contains("error", na=True)]
    df_answers.to_csv(path, index=False)
    parsed_answers = []
    for index, row in df_answers.iterrows():
        try:
            parsed_answer = json.loads(repair_json(row['Predicted_Answer']))
            assert type(parsed_answer) == dict
            parsed_answer = parsed_answer.get('answer', "None")
            if "no" in str(parsed_answer).lower():
                parsed_answer = {"Nobody"}
            if isinstance(parsed_answer, list):
                if len(parsed_answer) == 1:
                    parsed_answer = {parsed_answer[0]}
                else:
                    parsed_answer = {"Nobody"}
            parsed_answers.append(parsed_answer)
        except Exception as e:
            # print(f"Error parsing JSON in row {index} {row['Predicted_Answer']}: {e}")
            match = re.search(answer_pattern, row['Predicted_Answer'], flags=re.IGNORECASE)
            parsed_answers.append(match[0] if match else "None")
    df_answers['Final_Answer'] = parsed_answers
    # Merge with questions (assuming df_questions is already loaded)
    df_answers['hit'] = df_answers.apply(validate_and_compare, axis=1).astype(int)
    print(df_answers.shape)
    # Calculate hit rate by length
    model_name = "/".join(path.split("answers_")[-1].split(".csv")[0].split("_", 1))
    df_answers["model"] = model_name
    all_answers.append(df_answers)
    hit_rate = df_answers.groupby(['N_steps', 'Type'])['hit'].mean().sort_index().to_frame("hit")
    hit_rate["model"] = model_name
    heatmap_data.append(hit_rate.set_index(["model"], append=True))

heatmap_data = pd.concat(heatmap_data)
all_answers = pd.concat(all_answers)
heatmap_data["hit"] = heatmap_data["hit"].fillna(0) * 100

In [ ]:
all_answers[(all_answers['Type'].str.contains("count"))]['Answer'].value_counts()

In [ ]:
all_answers[(all_answers['Type'].str.contains("count"))]['Final_Answer'].value_counts()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from ast import parse
import re

# Define the regex pattern
answer_pattern = r'\b(Office|Garden|Kitchen|Bathroom|Hallway|Bedroom|\d)\b'

%config InlineBackend.figure_format='retina'

In [ ]:
heatmap_data = pd.read_csv("../results/newest_results.csv")
heatmap_data = heatmap_data.reset_index()
heatmap_data =heatmap_data.set_index(["seq_len", "qtype", "model"])

In [ ]:
# Plot heatmap with annotation
custom_cmap = sns.color_palette("RdYlGn", as_cmap=True)

plt.figure(figsize=(12, 8))
sns.heatmap((heatmap_data
             .groupby(['seq_len', 'qtype'])
             .mean()
             .reset_index()
             .pivot(index="seq_len", columns="qtype", values="hit")
            ), annot=True, fmt="2.0f", cmap=custom_cmap, vmin=0, vmax=100,  cbar_kws={'label': 'Accuracy'})
# plt.xlabel('Model')
plt.ylabel('Images in context')
plt.title("Mean of all models")
plt.tight_layout()
plt.xticks(rotation=-30)

plt.savefig(f"../results/mmlong_all_models.png", bbox_inches="tight")
plt.show()

In [ ]:
# Plot heatmap with annotation
custom_cmap = sns.color_palette("RdYlGn", as_cmap=True)

plt.figure(figsize=(12, 8))

pivot = (
    heatmap_data
     .groupby(['qtype', 'model'])
     .mean()
     .reset_index()
     .pivot(index="model", columns="qtype", values="hit")
)

# Sort the columns by their mean values
sorted_columns = pivot.mean().sort_values(ascending=False).index.sort_values()

pivot = pivot[sorted_columns]

sns.heatmap(pivot, annot=True, fmt="2.0f", cmap=custom_cmap,  vmin=0, vmax=100, cbar_kws={'label': 'Accuracy'})

plt.xlabel('Model')
plt.ylabel('Type of question')
plt.title("Mean of all lengths")
plt.tight_layout()
plt.xticks(rotation=90)
plt.yticks(rotation=-60)

plt.savefig(f"../results/mmlong_all_lengths.png", bbox_inches="tight")
plt.show()

In [ ]:
# Plot heatmap with annotation
custom_cmap = sns.color_palette("RdYlGn", as_cmap=True)

plt.figure(figsize=(12, 8))

pivot = (
    heatmap_data
     .groupby(['seq_len', 'model'])
     .mean()
     .reset_index()
     .pivot(index="seq_len", columns="model", values="hit")
)

# Sort the columns by their mean values
sorted_columns = pivot.mean().sort_values(ascending=False).index.sort_values()

pivot = pivot[sorted_columns]

sns.heatmap(pivot, annot=True, fmt="2.0f", cmap=custom_cmap,  vmin=0, vmax=100, cbar_kws={'label': 'Accuracy'})

plt.xlabel('Model')
plt.ylabel('Images in context')
plt.title("Mean of all questions")
plt.tight_layout()
plt.xticks(rotation=90)

plt.savefig(f"../results/mmlong_all_questions.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
for model in heatmap_data.index.get_level_values("model").unique():
    # Plot heatmap with annotation
    custom_cmap = sns.color_palette("RdYlGn", as_cmap=True)

    plt.figure(figsize=(16, 8))
    sns.heatmap((heatmap_data
                 .xs(model, level=2)
                 .reset_index()
                 .pivot(index="seq_len", columns="qtype", values="hit")
                ), annot=True, fmt="2.0f", cmap=custom_cmap, vmin=0, vmax=100, cbar_kws={'label': 'Accuracy'})
    # plt.xlabel('Model')
    plt.ylabel('Images in context')
    plt.title(model)
    plt.tight_layout()
    plt.xticks(rotation=90)
    
    plt.savefig(f"../results/mmlong_{"_".join(model.split("/"))}.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
%%time
import xgrammar as xgr

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoConfig

In [ ]:
# Get tokenizer info
model_id = "Qwen/Qwen2.5-VL-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
config = AutoConfig.from_pretrained(model_id)
# This can be larger than tokenizer.vocab_size due to paddings
full_vocab_size = config.vocab_size
tokenizer_info = xgr.TokenizerInfo.from_huggingface(tokenizer, vocab_size=full_vocab_size)

compiler = xgr.GrammarCompiler(tokenizer_info, max_threads=8)

In [ ]:
SEED = 0xBADFACE

WIDTH, HEIGHT = 512, 512

SEQ_LENGTHS = [1, 2, 4, 8, 16, 32, 64, 128]  # [2 ** i for i in range(8)]
N_QUESTIONS = 50

ROOMS: list[str] = ["Kitchen", "Bathroom", "Garden", "Office", "Bedroom", "Hallway"]
CHARS: list[str] = ["Sandra", "Mary", "John", "Daniel", "Michael"]
NOBODY: str = "Nobody"

AnswerTypePerson, AnswerTypeRoom, AnswerTypeNumber = "person", "room", "number"

BLUE = (0, 0, 255)
RED = (255, 0, 0)
GREEN = (0, 255, 0)
YELLOW = (255, 255, 0)
PURPLE = (128, 0, 128)
COLORS = [BLUE, RED, GREEN, YELLOW, PURPLE]


In [ ]:

from typing import List, Dict, Tuple, Union, Literal, Optional
from pydantic import BaseModel

class RoomAnswer(BaseModel):
    reasoning: Optional[str] = None
    answer: Literal[*ROOMS]


class NumberAnswer(BaseModel):
    reasoning: Optional[str] = None
    answer: int = 0


class PersonAnswer(BaseModel):
    reasoning: Optional[str] = None
    answer: Literal[*CHARS, NOBODY] = NOBODY


In [ ]:
number_grammar = compiler.compile_json_schema(NumberAnswer)
print(number_grammar.grammar._handle.to_string())

In [ ]:
room_grammar = compiler.compile_json_schema(RoomAnswer)
print(room_grammar.grammar._handle.to_string())

In [ ]:
person_grammar = compiled_grammar = compiler.compile_json_schema(PersonAnswer)
print(person_grammar.grammar._handle.to_string())

In [ ]:
gbs = heatmap_data.groupby('Type')

for name, group in gbs:
    # Plot heatmap with annotation
    custom_cmap = sns.color_palette("RdYlGn", as_cmap=True)
    
    plt.figure(figsize=(12, 8))
    sns.heatmap(group.reset_index().pivot(index="N_steps", columns="model", values="hit"), annot=True, fmt="2.0f", cmap=custom_cmap, cbar_kws={'label': 'Accuracy'})
    # plt.xlabel('Model')
    plt.ylabel('Images in context')
    plt.title(f"Question {name}")
    plt.tight_layout()
    plt.xticks(rotation=30)
    
    plt.savefig(f"results/mmlong_type_{name}.png", bbox_inches="tight")
    plt.show()

In [ ]:
heatmap_data.to_csv("results/newest_results.csv")

In [ ]:
opencompass_df = pd.read_csv("opencompass.csv", index_col=1)
opencompass_df

In [ ]:
bench_cols = ['Avg. Score', 'MMBench V1.1', 'MMStar', 'MMMU', 'MathVista', 'HallusionBench Avg.', 'AI2D', 'OCRBench', 'MMVet']

In [ ]:
opencompass_df = opencompass_df[bench_cols].dropna()

In [ ]:
opencompass_df.index = opencompass_df.index.str.replace('.', '_', regex=False)

In [ ]:
# Reset df1 index so we can work with 'model' as a column
df1_reset = heatmap_data.loc[(heatmap_data!=0).any(axis=1)].groupby(['N_steps', 'model'])['hit'].mean().reset_index()

# Function to find matching models in df2 based on substring
def find_model_match(row, df2):
    model = row['model'].split("/")[-1].strip("-Instruct")
    
    matches = []
    for idx in df2.index:
        if model in idx:
            matches.append(idx)
    if len(matches):
        return sorted(matches, key = lambda x: len(x))[0]
    return None

# Apply the function to create a new column in df1_reset that maps to df2's index
df1_reset['model_match'] = df1_reset.apply(lambda row: find_model_match(row, opencompass_df), axis=1)


In [ ]:
df1_reset = df1_reset[df1_reset['model'] != 'Qwen/QVQ-72B-Preview']

In [ ]:
df1_reset

In [ ]:
df1_reset[['model','model_match']].value_counts()

In [ ]:
result = pd.merge(df1_reset, opencompass_df, left_on='model_match', right_index=True, how='left')

# Drop the temporary 'model_df2' column if not needed
result = result.drop(columns=['model_match'])

In [ ]:
df = result

In [ ]:
import numpy as np
from scipy.stats import pearsonr

In [ ]:
df.columns

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Define the number of rows and columns for the subplots grid
n_rows = len(df['N_steps'].dropna().unique())  # One row per unique 'N_steps'
n_cols = len(columns_to_compare)               # One column per variable in 'columns_to_compare'

# Create a figure and axes for subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows), constrained_layout=True)

# Flatten the axes array for easy iteration if it's multidimensional
if n_rows > 1 and n_cols > 1:
    axes = axes.flatten()
else:
    axes = [axes]  # Ensure axes is iterable even if there's only one subplot

# Counter to keep track of which axis to plot on
plot_idx = 0

# Loop through each group based on 'N_steps'
for n_step, group in df.dropna().groupby('N_steps'):
    for col in columns_to_compare:
        # Plot on the current axis
        sns.regplot(data=group, x=col, y='hit', ax=axes[plot_idx], label=f'N_steps={n_step}')
        
        # Set title and labels for clarity
        axes[plot_idx].set_title(f'N_steps={n_step}, X={col}')
        axes[plot_idx].set_xlabel(col)
        axes[plot_idx].set_ylabel('hit')
        
        # Increment the plot index
        plot_idx += 1

# Save the entire figure to a single PNG file
plt.savefig('pairwise_plots.png', dpi=300, bbox_inches='tight')

# Show the plot (optional)
plt.show()

In [ ]:
df[df['N_steps'] == 128]

In [ ]:
pearson_df

In [ ]:
pearson_df

In [ ]:
# Columns to calculate Pearson correlation with 'hit'
columns_to_compare = df.columns[df.columns.get_loc('hit') + 1:]

# Group by 'N_steps' and calculate Pearson correlations
pearson_results = []
for n_step, group in df.dropna().groupby('N_steps'):
    correlations = {'N_steps': n_step}
    hits = group.groupby('model')['hit'].mean().to_numpy()
    for col in columns_to_compare: 
        bench = group.groupby('model')[col].mean().to_numpy()
        corr, _ = pearsonr(bench, hits)
        correlations[col] = corr
        correlations[col + "_n"] = len(bench)
    pearson_results.append(correlations)

# Convert results to a DataFrame
pearson_df = pd.DataFrame(pearson_results)


In [ ]:
cols = ['Avg. Score', 'MMMU', 'MMBench V1.1', 'AI2D']

In [ ]:
columns_to_compare

In [ ]:
# Plotting
colors = plt.cm.inferno(np.linspace(0.1, 0.9, len(cols)))
plt.figure(figsize=(12, 6))
for i, col in enumerate(cols):
    plt.plot(pearson_df['N_steps'], pearson_df[col], color=colors[i], label=col, marker='o')
    std = np.sqrt((1 - pearson_df[col] ** 2) / (pearson_df[col + "_n"] - 2))
    plt.fill_between(pearson_df['N_steps'], pearson_df[col] - std, (pearson_df[col] + std).clip(0, 1), color=colors[i], alpha=0.4)
plt.title('Pearson Correlation of "hit" vs Other Columns by N_steps')
plt.xlabel('N_steps')
plt.xscale("log", base=2)
plt.xticks(2 ** np.arange(0, 8))
plt.ylabel('Pearson Correlation')
plt.legend(title='Columns', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.savefig('corr.png', dpi=300)
plt.show()